# OneLake Shortcut Automation via Fabric CLI (`fab`)

Create OneLake shortcuts in bulk from a manifest, using the **Microsoft Fabric CLI**
(`fab`) instead of calling the REST API directly.

Reference: https://microsoft.github.io/fabric-cli/examples/shortcut_examples/

**What this notebook does**
1. Installs / verifies `fabric-cli`.
2. Authenticates `fab` (interactive or service principal).
3. Reads a manifest (table name + target path).
4. For each row, runs `fab ln ...` to create the shortcut.
5. Verifies with `fab exists` and captures per-row results.

**Supported target types shown**
- `oneLake`  — internal Fabric-to-Fabric shortcut
- `adlsGen2` — Azure Data Lake Storage Gen2
- `gcs`      — Google Cloud Storage (HMAC connection)
- `s3`       — Amazon S3

Switch the `TARGET_TYPE` variable below to pick the behavior for your manifest.


In [ ]:
# Install / upgrade the Fabric CLI
# (safe to re-run; no-op if already installed)
%pip install --quiet --upgrade ms-fabric-cli


In [ ]:
import json
import os
import shlex
import subprocess
from typing import Any, Dict, List

import pandas as pd


def run_fab(args: str, check: bool = False) -> subprocess.CompletedProcess:
    """Run a `fab` command and return the CompletedProcess."""
    cmd = f"fab {args}"
    proc = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True,
    )
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{proc.stderr}")
    return proc


# Verify installation
ver = run_fab("--version")
print(ver.stdout or ver.stderr)


## Authenticate `fab`

Pick one of the options below.

- **Interactive (recommended for local use):**  run `!fab auth login` in a terminal.
- **Service principal (headless / CI):** set env vars and run `fab auth login` with flags.

See: https://microsoft.github.io/fabric-cli/examples/auth_examples/


In [ ]:
# Option A: interactive browser login (run once per machine)
# !fab auth login

# Option B: service principal (uncomment and fill in)
# os.environ["FAB_TENANT_ID"]     = "<tenant-guid>"
# os.environ["FAB_CLIENT_ID"]     = "<sp-app-id>"
# os.environ["FAB_CLIENT_SECRET"] = "<sp-secret>"
# !fab auth login -u $FAB_CLIENT_ID -p $FAB_CLIENT_SECRET --tenant $FAB_TENANT_ID

# Confirm auth state
print(run_fab("auth status").stdout)


In [ ]:
# =========================
# Configuration
# =========================
# Workspace and lakehouse as referenced by the Fabric CLI path syntax.
# Example path: ws1.Workspace/lh1.Lakehouse/Tables/<name>.Shortcut
WORKSPACE   = "ws1.Workspace"
LAKEHOUSE   = "lh1.Lakehouse"
PARENT_DIR  = "Tables"        # "Tables" or "Files" (or "Files/sub")

# Target type: "gcs" | "adlsGen2" | "s3" | "oneLake"
TARGET_TYPE = "gcs"

# Shared settings for external targets
CONNECTION_ID = "<connection-guid>"                  # pre-created Fabric connection
GCS_LOCATION  = "https://storage.googleapis.com"     # or https://<bucket>.storage.googleapis.com
ADLS_LOCATION = "https://<account>.dfs.core.windows.net/"
S3_LOCATION   = "https://<bucket>.s3.<region>.amazonaws.com"

# Safety
DRY_RUN = True
FORCE   = True   # pass -f to skip confirmation prompts


In [ ]:
# =========================
# Manifest of shortcuts to create
# =========================
# Columns:
#   name      - shortcut name (the '<name>.Shortcut' leaf in the lakehouse path)
#   bucket    - GCS/S3 bucket or ADLS container (ignored for oneLake)
#   subpath   - path within the bucket/container (no leading slash needed)
#   onelake_target - for TARGET_TYPE="oneLake": relative target path for `fab ln --target`
#
# For GCS/S3 the final "subpath" sent to the CLI will be "/<bucket>/<subpath>".

manifest_rows: List[Dict[str, str]] = [
    {"name": "customers", "bucket": "my-gcs-bucket", "subpath": "bronze/customers"},
    {"name": "orders",    "bucket": "my-gcs-bucket", "subpath": "bronze/orders"},
    {"name": "products",  "bucket": "my-gcs-bucket", "subpath": "bronze/products"},
]

# Optional: load from CSV instead
# manifest_rows = pd.read_csv("shortcut_manifest.csv").to_dict(orient="records")

manifest_df = pd.DataFrame(manifest_rows)
manifest_df


In [ ]:
# =========================
# Build the `fab ln` command for a manifest row
# =========================
def shortcut_path(name: str) -> str:
    """Fabric CLI path for the shortcut leaf."""
    return f"{WORKSPACE}/{LAKEHOUSE}/{PARENT_DIR}/{name}.Shortcut"


def build_ln_command(row: Dict[str, Any]) -> str:
    """Return the `fab ln` command (without the leading 'fab ')."""
    path = shortcut_path(row["name"])
    force_flag = " -f" if FORCE else ""

    if TARGET_TYPE == "oneLake":
        target = row["onelake_target"]
        return f"ln {shlex.quote(path)} --type oneLake --target {shlex.quote(target)}{force_flag}"

    if TARGET_TYPE == "gcs":
        payload = {
            "location": GCS_LOCATION,
            "subpath": f"/{row['bucket']}/{row['subpath'].lstrip('/')}",
            "connectionId": CONNECTION_ID,
        }
        return f"ln {shlex.quote(path)} --type gcs -i {shlex.quote(json.dumps(payload))}{force_flag}"

    if TARGET_TYPE == "adlsGen2":
        payload = {
            "location": ADLS_LOCATION,
            "subpath": f"{row['bucket']}/{row['subpath'].lstrip('/')}",
            "connectionId": CONNECTION_ID,
        }
        return f"ln {shlex.quote(path)} --type adlsGen2 -i {shlex.quote(json.dumps(payload))}{force_flag}"

    if TARGET_TYPE == "s3":
        payload = {
            "location": S3_LOCATION,
            "subpath": f"/{row['subpath'].lstrip('/')}",
            "connectionId": CONNECTION_ID,
        }
        return f"ln {shlex.quote(path)} --type s3 -i {shlex.quote(json.dumps(payload))}{force_flag}"

    raise ValueError(f"Unsupported TARGET_TYPE: {TARGET_TYPE}")


# Preview the commands without running them
for r in manifest_df.to_dict(orient="records"):
    print("fab " + build_ln_command(r))


In [ ]:
# =========================
# Execute: create shortcuts sequentially via `fab ln`
# =========================
results: List[Dict[str, Any]] = []

for row in manifest_df.to_dict(orient="records"):
    name = row["name"]
    path = shortcut_path(name)
    cmd_args = build_ln_command(row)

    if DRY_RUN:
        print(f"[DRY_RUN] fab {cmd_args}")
        results.append({"name": name, "status": "DRY_RUN", "returncode": None, "message": ""})
        continue

    print(f"[RUN] fab {cmd_args}")
    proc = run_fab(cmd_args)
    status = "SUCCESS" if proc.returncode == 0 else "FAILED"
    message = (proc.stdout or "") + (proc.stderr or "")
    results.append({
        "name": name,
        "status": status,
        "returncode": proc.returncode,
        "message": message.strip()[:500],
    })
    print(f"  -> {status} (rc={proc.returncode})")


In [ ]:
# =========================
# Verify created shortcuts with `fab exists` and `fab get`
# =========================
verification: List[Dict[str, Any]] = []

for row in manifest_df.to_dict(orient="records"):
    path = shortcut_path(row["name"])
    exists_proc = run_fab(f"exists {shlex.quote(path)}")
    exists = (exists_proc.stdout or "").strip().lower().startswith("true") or exists_proc.returncode == 0

    target = ""
    if exists and not DRY_RUN:
        get_proc = run_fab(f"get {shlex.quote(path)} -q target")
        target = (get_proc.stdout or "").strip()

    verification.append({"name": row["name"], "path": path, "exists": exists, "target": target[:300]})

verification_df = pd.DataFrame(verification)
verification_df


In [ ]:
# Results summary
results_df = pd.DataFrame(results)
results_df


## Cleanup (optional)

Remove all shortcuts created by this manifest. Uses `fab rm -f`.


In [ ]:
# Uncomment to remove the shortcuts defined in the manifest
# for row in manifest_df.to_dict(orient="records"):
#     path = shortcut_path(row["name"])
#     proc = run_fab(f"rm {shlex.quote(path)} -f")
#     print(path, "->", proc.returncode, (proc.stdout or proc.stderr).strip())


## Runbook

1. `%pip install ms-fabric-cli` (cell 2).
2. Authenticate: `!fab auth login` (interactive) or service principal env vars.
3. Set `WORKSPACE`, `LAKEHOUSE`, `PARENT_DIR`, `TARGET_TYPE`, `CONNECTION_ID`.
4. Populate `manifest_rows`.
5. `DRY_RUN = True` to preview the generated `fab ln` commands.
6. `DRY_RUN = False` to execute sequentially.
7. Review `results_df` and `verification_df`; rerun failed rows as needed.

**CLI commands used** (see [Fabric CLI shortcut examples](https://microsoft.github.io/fabric-cli/examples/shortcut_examples/))
- Create:  `fab ln <path>.Shortcut --type <type> -i '<json>'`
- Verify:  `fab exists <path>.Shortcut`
- Inspect: `fab get <path>.Shortcut -q target`
- Delete:  `fab rm <path>.Shortcut -f`
